<a href="https://colab.research.google.com/github/elkins/synth-afm/blob/main/examples/tip_geometry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AFM Tip Geometry & Scanning Artifacts

This tutorial explores how the physical shape of an Atomic Force Microscopy (AFM) cantilever tip affects the resulting topographical image. Because the tip has a finite radius, biological samples appear wider than they actually are (tip convolution or dilation).

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q synth-afm biotite matplotlib
else:
    import os
    sys.path.insert(0, os.path.abspath(".."))

## 1. Load a Target Molecule
We'll generate a topographical map of a small DNA double helix (or any protein).

In [ ]:
import biotite.structure.io as strucio
import biotite.database.rcsb as rcsb

# Download a DNA structure (e.g., 1BNA)
pdb_file = rcsb.fetch("1BNA", "pdb", ".")
structure = strucio.load_structure(pdb_file)
coords = structure.coord

## 2. Simulate AFM Scans
We will simulate scanning the molecule using the `AFMSimulator`. We will compare an "ideal" infinitesimally sharp tip against a realistic "blunt" tip.

In [ ]:
from synth_afm import AFMSimulator

# Define a 100x100 grid spanning 60 Ångstroms
sim_sharp = AFMSimulator(grid_size=(100, 100), pixel_size=0.6, tip_radius=1.0)
sim_blunt = AFMSimulator(grid_size=(100, 100), pixel_size=0.6, tip_radius=10.0)

# Simulate Sharp Tip (1.0 Å radius)
height_map_sharp = sim_sharp.scan(coords)

# Simulate Blunt Tip (10.0 Å radius)
height_map_blunt = sim_blunt.scan(coords)

## 3. Visualize Tip Convolution
Let's plot both height maps side-by-side. Notice how the blunt tip "smears" the structural features, making the DNA helix look artificially wide and obscuring the major/minor grooves.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Sharp Tip
im1 = ax1.imshow(height_map_sharp, cmap='afmhot', origin='lower')
ax1.set_title('Sharp Tip (R = 1.0 Å)')
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Blunt Tip
im2 = ax2.imshow(height_map_blunt, cmap='afmhot', origin='lower')
ax2.set_title('Blunt Tip (R = 10.0 Å)')
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

# Difference Map
diff = height_map_blunt - height_map_sharp
im3 = ax3.imshow(diff, cmap='viridis', origin='lower')
ax3.set_title('Tip Dilation Artifact')
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()